In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, classification_report, confusion_matrix
import seaborn as sns

# ---------------------------------------------------------
# 1. LOAD & CLEAN DATA
# ---------------------------------------------------------
plant1_generation_path = 'Plant_1_Generation_Data.csv'
plant1_weather_path = 'Plant_1_Weather_Sensor_Data.csv'

try:
    df_gen1 = pd.read_csv(plant1_generation_path)
    df_gen1['DATE_TIME'] = pd.to_datetime(df_gen1['DATE_TIME'], errors='coerce', dayfirst=True)
    df_gen1.dropna(subset=['DATE_TIME'], inplace=True)
except FileNotFoundError:
    df_gen1 = pd.DataFrame()

try:
    df_weather1 = pd.read_csv(plant1_weather_path)
    df_weather1['DATE_TIME'] = pd.to_datetime(df_weather1['DATE_TIME'], errors='coerce')
    df_weather1.dropna(subset=['DATE_TIME'], inplace=True)
except FileNotFoundError:
    df_weather1 = pd.DataFrame()

def clean_dataframe(df):
    numeric_cols = df.select_dtypes(include=np.number).columns
    if not numeric_cols.empty:
        df[numeric_cols] = df[numeric_cols].interpolate(method='linear', limit_direction='both')
        df[numeric_cols] = df[numeric_cols].ffill().bfill()
    return df

if not df_gen1.empty:
    df_gen1 = clean_dataframe(df_gen1.copy())
if not df_weather1.empty:
    df_weather1 = clean_dataframe(df_weather1.copy())

# ---------------------------------------------------------
# 2. AGGREGATE TO PLANT LEVEL
# ---------------------------------------------------------
# Aggregate generation data (summing DC_POWER across all inverters)
df_gen1_agg = df_gen1.groupby('DATE_TIME').agg({
    'DC_POWER': 'sum',
    'AC_POWER': 'sum',
    'TOTAL_YIELD': 'max'
}).reset_index()

# Weather sensors are plant-wide; average readings across duplicate timestamps
df_weather1_agg = df_weather1.groupby('DATE_TIME').agg({
    'AMBIENT_TEMPERATURE': 'mean',
    'MODULE_TEMPERATURE': 'mean',
    'IRRADIATION': 'mean'
}).reset_index()

# Merge aggregated generation and weather data
merged_plant1 = pd.merge(
    df_gen1_agg,
    df_weather1_agg,
    on='DATE_TIME',
    how='inner'
)

# ---------------------------------------------------------
# 3. TRAIN PLANT-LEVEL LINEAR REGRESSION MODEL
# ---------------------------------------------------------
model_features = ['IRRADIATION', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE']
X = merged_plant1[model_features].dropna()
y = merged_plant1.loc[X.index, 'DC_POWER']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model_plant1 = LinearRegression()
model_plant1.fit(X_train, y_train)

y_pred = model_plant1.predict(X_test)
print(f"Model R² Score: {r2_score(y_test, y_pred):.4f}")

# ---------------------------------------------------------
# 4. DEFINE ALERT LOGIC
# ---------------------------------------------------------
def generate_ml_maintenance_alerts(df, model, features, prediction_threshold_ratio=0.85, min_irradiation=0.05):
    df_alerts = df.copy()
    df_alerts['ML_ALERT'] = False

    # Only evaluate during stable daylight hours (filtering out sunrise/sunset transition noise)
    active_indices = df_alerts[df_alerts['IRRADIATION'] >= min_irradiation].index

    if not active_indices.empty:
        X_predict = df_alerts.loc[active_indices, features]
        df_alerts.loc[active_indices, 'PREDICTED_DC_POWER'] = model.predict(X_predict)

        condition_alert = (
            (df_alerts['DC_POWER'] < prediction_threshold_ratio * df_alerts['PREDICTED_DC_POWER']) &
            (df_alerts['DC_POWER'] > 0)
        )
        df_alerts.loc[condition_alert, 'ML_ALERT'] = True
    else:
        df_alerts['PREDICTED_DC_POWER'] = 0

    return df_alerts

# ---------------------------------------------------------
# 5. CREATE SYNTHETIC GROUND-TRUTH TEST DATASET
# ---------------------------------------------------------
df_synth = merged_plant1.copy()
df_synth['TRUE_ANOMALY'] = False

# Inject Anomaly 1: Sudden 40% efficiency drop over 5 days (May 20 - May 25)
anomaly_mask_1 = (df_synth['DATE_TIME'] >= '2020-05-20') & (df_synth['DATE_TIME'] <= '2020-05-25') & (df_synth['IRRADIATION'] > 0)
df_synth.loc[anomaly_mask_1, 'DC_POWER'] *= 0.60
df_synth.loc[anomaly_mask_1, 'TRUE_ANOMALY'] = True

# Inject Anomaly 2: 25% degradation over 3 days (June 1 - June 3)
anomaly_mask_2 = (df_synth['DATE_TIME'] >= '2020-06-01') & (df_synth['DATE_TIME'] <= '2020-06-03') & (df_synth['IRRADIATION'] > 0)
df_synth.loc[anomaly_mask_2, 'DC_POWER'] *= 0.75
df_synth.loc[anomaly_mask_2, 'TRUE_ANOMALY'] = True

# ---------------------------------------------------------
# 6. RUN MODEL & EVALUATE
# ---------------------------------------------------------
df_synth_eval = generate_ml_maintenance_alerts(df_synth, model_plant1, model_features)

# Filter evaluation to active daylight hours
active_eval = df_synth_eval[df_synth_eval['IRRADIATION'] > 0].copy()
active_eval['EXPECTED_ALERT_NUM'] = active_eval['TRUE_ANOMALY'].astype(int)
active_eval['MODEL_ALERT_NUM'] = active_eval['ML_ALERT'].astype(int)

print("\n--- Classification Report: Ground Truth vs Model Alerts ---")
print(classification_report(active_eval['TRUE_ANOMALY'], active_eval['ML_ALERT'], target_names=['Normal State', 'Maintenance Required']))

# ---------------------------------------------------------
# 7. VISUALIZE RESULTS
# ---------------------------------------------------------
sample_period = active_eval[
    (active_eval['DATE_TIME'] >= '2020-05-18') &
    (active_eval['DATE_TIME'] <= '2020-06-05')
].sort_values('DATE_TIME')

plt.figure(figsize=(14, 5))
plt.step(sample_period['DATE_TIME'], sample_period['EXPECTED_ALERT_NUM'],
         where='mid', label='Expected Alert (Ground Truth)', color='red', linewidth=2)
plt.step(sample_period['DATE_TIME'], sample_period['MODEL_ALERT_NUM'] * 0.9,
         where='mid', label='Model ML Alert Flagged', color='black', linestyle='--', linewidth=1.5)

plt.title('Corrected Alert Comparison (Plant-Level Aggregated)')
plt.xlabel('Date / Time')
plt.ylabel('Alert Active (1 = Yes, 0 = No)')
plt.yticks([0, 1], ['Normal (0)', 'Alert (1)'])
plt.ylim(-0.1, 1.2)
plt.legend(loc='upper right')
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()